5 Задание
1 датасет

Обучить классификатор, протестировать его работу, разобраться какие есть параметры и посмотреть меняет ли что-то их настройка.

Качество работы классификатора оцениваем с помощью F1-score (mocro, macro, avg. - в чем разница – разобраться самостоятельно в качестве задания)

Классификаторы:
🔸Decision Tree
🔸Random Forest
🔸Gradient Boosting Machines (GBM)
🔸AdaBoost (Адаптивный бустинг)

Для работы с текстами, тексты представляются в векторной форме:
1) предобработка текста - с лемматизацией, без, со стеммингом + tf-idf как вес слова в тексте при построении векторов для документов.

2) лемматизация:
+3 варианта векторного представления:
- 0,1 есть слово в документе/нет
- 0, частота слова в документе
- tf-idf (из пункта 1)

3) лемматизация + оставить только сущ. и прил.:
+ 3 варианта векторного представления:
- 0,1 есть слово в документе/нет
- 0, частота слова в документе
- tf-idf (из пункта 1)

Смотрим какие отличия. Для комбинации, которая кажется оптимальной крутим параметры алгоритма классификации и смотрим, что происходит.

Результаты собрать в таблички для презентации.

In [10]:
import nltk
import string
from datasets import load_dataset
from nltk.corpus import wordnet, stopwords
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
import numpy as np

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('omw-1.4')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.8 MB/s eta 0:00:00


In [4]:
import re
import emoji
from collections import Counter

In [ ]:
dataset = load_dataset('emotion')
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])
validation_df = pd.DataFrame(dataset['validation'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
X_train = list(df_train['text']) + list(validation_df['text'])
y_train = list(df_train['label']) + list(validation_df['label'])
X_test = list(df_test['text'])
y_test = list(df_test['label'])

In [8]:
class Prepr:
  def  __init__(self,p_type='lemmatization',vectorizer='tf-idf',  filter_pos=None):
    self.p_type = p_type
    self.vectorizer = vectorizer
    self.filter_pos = filter_pos
    self.lemmatizor =  nltk.WordNetLemmatizer()
    self.stemmer = nltk.PorterStemmer()
    self.tf_idf = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1, 2))
    self.bin_vec = CountVectorizer(binary=True)
    self.freq_vec = CountVectorizer()
    self.is_fitted = False
    self.stop_words = set(stopwords.words('english'))
  def clean(self, tt):
    t = [self.clean_text(to) for to in tt]
    if self.p_type == 'lemmatization':
      text = self.preprocess_with_lemmatization(t)
    else:
      text = self.preprocess_with_stemming(t)
    return text
  def fit(self,tt):
    text = self.clean(tt)
    if self.vectorizer == 'tf-idf':
      matrix = self.tf_idf.fit(text)
      feature_names = self.tf_idf.get_feature_names_out()
    elif self.vectorizer == 'binary':
      matrix = self.bin_vec.fit(text)
      feature_names = self.bin_vec.get_feature_names_out()
    elif self.vectorizer == 'frequency':
      matrix = self.freq_vec.fit(text)
      feature_names = self.freq_vec.get_feature_names_out()
    self.is_fitted =True
    return self
  def transform(self, tt):
    text = self.clean(tt)
    if self.vectorizer == 'tf-idf':
      matrix = self.tf_idf.transform(text)
      feature_names = self.tf_idf.get_feature_names_out()
    elif self.vectorizer == 'binary':
      matrix = self.bin_vec.transform(text)
      feature_names = self.bin_vec.get_feature_names_out()
    elif self.vectorizer == 'frequency':
      matrix = self.freq_vec.transform(text)
      feature_names = self.freq_vec.get_feature_names_out()
    df = pd.DataFrame(matrix.toarray(), columns=feature_names)
    return df
  def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)
  def preprocess_with_lemmatization(self,text):
      docs = []
      for t in text:
          tt = t
          tt = tt.translate(str.maketrans('','',string.punctuation))
          tokens = nltk.word_tokenize(tt)
          if self.filter_pos:
            tagged = nltk.pos_tag(tokens)
            lemmatized_tokens = []
            for token, tag in tagged:
              if any(tag.startswith(pos) for pos in self.filter_pos):
                lemmatized = self.lemmatizor.lemmatize(token, self.get_wordnet_pos(tag))
                if lemmatized and lemmatized not in self.stop_words:
                  lemmatized_tokens.append(lemmatized)
          else:
            tagged = nltk.pos_tag(tokens)
            lemmatized_tokens = []
            for token, tag in tagged:
              lemmatized = self.lemmatizor.lemmatize(token, self.get_wordnet_pos(tag))
              if lemmatized and lemmatized not in self.stop_words:
                  lemmatized_tokens.append(lemmatized)
          doc = ' '.join(lemmatized_tokens)
          docs.append(doc)
      return docs
  def preprocess_with_stemming(self,text):
      docs = []
      for t in text:
          tt = t
          tt = tt.translate(str.maketrans('','',string.punctuation))
          tokens = nltk.word_tokenize(tt)
          if self.filter_pos:
            tagged = nltk.pos_tag(tokens)
            stemmed_tokens = []
            for token, tag in tagged:
              if any(tag.startswith(pos) for pos in self.filter_pos):
                stemmed = self.stemmer.stem(token)
                if stemmed and stemmed not in self.stop_words:
                  stemmed_tokens.append(stemmed)
          else:
            stemmed_tokens = []
            for token in tokens:
              stemmed = self.stemmer.stem(token)
              if stemmed and stemmed not in self.stop_words:
                  stemmed_tokens.append(stemmed)
          doc = ' '.join(stemmed_tokens)
          docs.append(doc)
      return docs
  def get_wordnet_pos(seelf, tag):
      if tag.startswith('J'):
          return wordnet.ADJ
      elif tag.startswith('V'):
          return wordnet.VERB
      elif tag.startswith('R'):
          return wordnet.ADV
      else:
          return wordnet.NOUN
  def clean_text(self, text):
        text = text.lower()
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
  def __call__(self, doc):
    return self.transform([doc]) if self.is_fitted else self.fit_transform([doc])

In [ ]:
prepr11 = Prepr()
result11 = prepr11(X_train)
prepr12 = Prepr(p_type='stemming')
result12 = prepr12(X_train)

NameError: name 'Prepr' is not defined

In [ ]:
result11.head()

,able,actually,ask,away,bad,bit,blog,book,care,child,...,use,wake,walk,want,way,week,work,world,write,year
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.980026,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
result12.head()

,actual,alway,amaz,ani,anyth,away,becaus,befor,better,bit,...,wa feel,want,way,week,whi,wonder,work,world,write,year
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
prepr21 = Prepr(p_type='lemmatization',vectorizer='binary')
prepr22 = Prepr(p_type='lemmatization',vectorizer='frequency')
result21 = prepr21(X_train)
result22 = prepr22(X_train)

In [ ]:
result21.head()

,aa,aac,aahh,aand,aaron,ab,abandon,abandonment,abate,abbigail,...,zombies,zone,zonisamide,zoo,zoom,zq,zucchini,zum,zumba,zz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
result22['humiliate']

,humiliate
0,1
1,0
2,0
3,0
4,0
...,...
17995,0
17996,0
17997,0
17998,0


In [ ]:
X_train[0]

'i didnt feel humiliated'

In [ ]:
prepr31 = Prepr(filter_pos=['N','J'])
prepr32 = Prepr(vectorizer='binary',filter_pos=['N','J'])
prepr33 = Prepr(vectorizer='frequency',filter_pos=['N','J'])
result31 = prepr31(X_train)
result32 = prepr32(X_train)
result33 = prepr33(X_train)

In [ ]:
result31.head()

,able,angry,anxious,bad,big,bit,blog,body,book,child,...,time,today,way,week,weird,woman,word,work,world,year
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
result32.head()

,aa,aac,aand,aaron,ab,abandonment,abbigail,abc,abdomen,abdominal,...,zoll,zombie,zone,zonisamide,zoo,zoom,zq,zucchini,zumba,zz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
result33.head()

,aa,aac,aand,aaron,ab,abandonment,abbigail,abc,abdomen,abdominal,...,zoll,zombie,zone,zonisamide,zoo,zoom,zq,zucchini,zumba,zz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


1 Модель

In [13]:
dataset = load_dataset("emotion")
train_texts = list(dataset['train']['text'])
train_labels = list(dataset['train']['label'])
test_texts = list(dataset['test']['text'])
test_labels = list(dataset['test']['label'])
X_train, X_val, y_train, y_val = train_test_split(
    train_texts, train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)
processed_data = {}

prepr_11 = Prepr(p_type='lemmatization', vectorizer='tf-idf', filter_pos=None)
X_train_11 = prepr_11.fit_transform(X_train)
X_val_11 = prepr_11.transform(X_val)
X_test_11 = prepr_11.transform(test_texts)

prepr_12 = Prepr(p_type='stemming', vectorizer='tf-idf', filter_pos=None)
X_train_12 = prepr_12.fit_transform(X_train)
X_val_12 = prepr_12.transform(X_val)
X_test_12 = prepr_12.transform(test_texts)

prepr_21 = Prepr(p_type='lemmatization', vectorizer='binary', filter_pos=None)
X_train_21 = prepr_21.fit_transform(X_train)
X_val_21 = prepr_21.transform(X_val)
X_test_21 = prepr_21.transform(test_texts)

prepr_22 = Prepr(p_type='lemmatization', vectorizer='frequency', filter_pos=None)
X_train_22 = prepr_22.fit_transform(X_train)
X_val_22 = prepr_22.transform(X_val)
X_test_22 = prepr_22.transform(test_texts)

prepr_31 = Prepr(p_type='lemmatization', vectorizer='tf-idf', filter_pos=['N', 'J'])
X_train_31 = prepr_31.fit_transform(X_train)
X_val_31 = prepr_31.transform(X_val)
X_test_31 = prepr_31.transform(test_texts)

prepr_32 = Prepr(p_type='lemmatization', vectorizer='binary', filter_pos=['N', 'J'])
X_train_32 = prepr_32.fit_transform(X_train)
X_val_32 = prepr_32.transform(X_val)
X_test_32 = prepr_32.transform(test_texts)

prepr_33 = Prepr(p_type='lemmatization', vectorizer='frequency', filter_pos=['N', 'J'])
X_train_33 = prepr_33.fit_transform(X_train)
X_val_33 = prepr_33.transform(X_val)
X_test_33 = prepr_33.transform(test_texts)
def model_train(X_train, y_train, X_val, y_val):
    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    results = {}

    dt = DecisionTreeClassifier(
        class_weight='balanced',
        max_depth=20,
        min_samples_leaf=20,
        min_samples_split=100,
        random_state=42
    )
    dt.fit(X_train_scaled, y_train)
    y_pred_dt = dt.predict(X_val_scaled)
    results['DecisionTree'] = {
        'macro': f1_score(y_val, y_pred_dt, average='macro'),
        'micro': f1_score(y_val, y_pred_dt, average='micro'),
        'weighted': f1_score(y_val, y_pred_dt, average='weighted')
    }
    rf = RandomForestClassifier(
        class_weight='balanced',
        max_depth=None,
        min_samples_leaf=10,
        min_samples_split=20,
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_scaled, y_train)
    y_pred_rf = rf.predict(X_val_scaled)
    results['RandomForest'] = {
        'macro': f1_score(y_val, y_pred_rf, average='macro'),
        'micro': f1_score(y_val, y_pred_rf, average='micro'),
        'weighted': f1_score(y_val, y_pred_rf, average='weighted')
    }

    gb = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        random_state=42,
        eval_metric='mlogloss'
    )
    gb.fit(X_train_scaled, y_train)
    y_pred_gb = gb.predict(X_val_scaled)
    results['XGBoost'] = {
        'macro': f1_score(y_val, y_pred_gb, average='macro'),
        'micro': f1_score(y_val, y_pred_gb, average='micro'),
        'weighted': f1_score(y_val, y_pred_gb, average='weighted')
    }
    base_estimator = LogisticRegression(
    max_iter=100,
    class_weight='balanced',
    random_state=42,
    C=1.0
    )

    return results
method_descriptions = {
    '11': 'Лемматизация + TF-IDF',
    '12': 'Стемминг + TF-IDF',
    '21': 'Лемматизация + Binary',
    '22': 'Лемматизация + Frequency',
    '31': 'Лемматизация + сущ./прил. + TF-IDF',
    '32': 'Лемматизация + сущ./прил. + Binary',
    '33': 'Лемматизация + сущ./прил. + Frequency'
}
data_dict = {
    '11': (X_train_11, X_val_11, X_test_11),
    '12': (X_train_12, X_val_12, X_test_12),
    '21': (X_train_21, X_val_21, X_test_21),
    '22': (X_train_22, X_val_22, X_test_22),
    '31': (X_train_31, X_val_31, X_test_31),
    '32': (X_train_32, X_val_32, X_test_32),
    '33': (X_train_33, X_val_33, X_test_33)
}

In [14]:
all_results = []

for method_name, (X_train_data, X_val_data, X_test_data) in data_dict.items():
    print(f"\n{method_name}: {method_descriptions[method_name]}")

    results = model_train(X_train_data, y_train, X_val_data, y_val)

    for model_name, metrics in results.items():
        all_results.append({
            'Method': method_name,
            'Description': method_descriptions[method_name],
            'Model': model_name,
            'F1_macro': metrics['macro'],
            'F1_micro': metrics['micro'],
            'F1_weighted': metrics['weighted']
        })
        print(f"  {model_name}: macro={metrics['macro']:.4f}, micro={metrics['micro']:.4f}")

df_results = pd.DataFrame(all_results)

pivot_micro = df_results.pivot_table(
    values='F1_micro',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

pivot_macro = df_results.pivot_table(
    values='F1_macro',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

pivot_weighted = df_results.pivot_table(
    values='F1_weighted',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

print("\nF1 MICRO:")
print(pivot_micro)

print("\nF1 MACRO:")
print(pivot_macro)

print("\nF1 WEIGHTED:")
print(pivot_weighted)

best_micro = df_results.loc[df_results.groupby('Model')['F1_micro'].idxmax()]

for _, row in best_micro.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  Метод: {row['Description']}")
    print(f"  F1_micro: {row['F1_micro']:.4f}")
    print(f"  F1_macro: {row['F1_macro']:.4f}")

for model_name in ['DecisionTree', 'RandomForest', 'XGBoost', 'AdaBoost']:
    best_row = df_results[df_results['Model'] == model_name].loc[
        df_results[df_results['Model'] == model_name]['F1_micro'].idxmax()
    ]
    best_method = best_row['Method']
    best_desc = best_row['Description']
    print(f"\n{model_name} (лучший: {best_desc})")
    X_train_full = data_dict[best_method][0]
    X_test_data = data_dict[best_method][2]

    from scipy.sparse import vstack
    X_train_combined = vstack([X_train_full, data_dict[best_method][1]])
    y_train_combined = np.concatenate([y_train, y_val])

    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train_combined)
    X_test_scaled = scaler.transform(X_test_data)

    if model_name == 'DecisionTree':
        model = DecisionTreeClassifier(
            class_weight='balanced',
            max_depth=20,
            min_samples_leaf=20,
            min_samples_split=100,
            random_state=42
        )
    elif model_name == 'RandomForest':
        model = RandomForestClassifier(
            class_weight='balanced',
            max_depth=None,
            min_samples_leaf=10,
            min_samples_split=20,
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    elif model_name == 'XGBoost':
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0,
            reg_alpha=0,
            reg_lambda=1,
            random_state=42,
            eval_metric='mlogloss'
        )

    model.fit(X_train_scaled, y_train_combined)
    y_pred_test = model.predict(X_test_scaled)

    test_micro = f1_score(test_labels, y_pred_test, average='micro')
    test_macro = f1_score(test_labels, y_pred_test, average='macro')

    print(f"  Test F1_micro: {test_micro:.4f}")
    print(f"  Test F1_macro: {test_macro:.4f}")


11: Лемматизация + TF-IDF


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:20:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  DecisionTree: macro=0.2311, micro=0.1903
  RandomForest: macro=0.7329, micro=0.7600
  XGBoost: macro=0.7422, micro=0.7666

12: Стемминг + TF-IDF


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:23:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  DecisionTree: macro=0.3060, micro=0.3613
  RandomForest: macro=0.7623, micro=0.7853
  XGBoost: macro=0.7677, micro=0.7947

21: Лемматизация + Binary


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


  Test F1_micro: 0.8220
  Test F1_macro: 0.7788

XGBoost (лучший: Лемматизация + Binary)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


  Test F1_micro: 0.1375
  Test F1_macro: 0.0403


ValueError: attempt to get argmax of an empty sequence

In [17]:
dataset = load_dataset("emotion")
train_texts = list(dataset['train']['text'])
train_labels = list(dataset['train']['label'])
test_texts = list(dataset['test']['text'])
test_labels = list(dataset['test']['label'])
X_train, X_val, y_train, y_val = train_test_split(
    train_texts, train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)
processed_data = {}
prepr_11 = Prepr(p_type='lemmatization', vectorizer='tf-idf', filter_pos=None)
X_train_11 = prepr_11.fit_transform(X_train)
X_val_11 = prepr_11.transform(X_val)
X_test_11 = prepr_11.transform(test_texts)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_11 = svd.fit_transform(X_train_11)
X_test_11 = svd.transform(X_test_11)
X_val_11 = svd.transform(X_val_11)

prepr_12 = Prepr(p_type='stemming', vectorizer='tf-idf', filter_pos=None)
X_train_12 = prepr_12.fit_transform(X_train)
X_val_12 = prepr_12.transform(X_val)
X_test_12 = prepr_12.transform(test_texts)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_12 = svd.fit_transform(X_train_12)
X_test_12 = svd.transform(X_test_12)
X_val_12 = svd.transform(X_val_12)

prepr_21 = Prepr(p_type='lemmatization', vectorizer='binary', filter_pos=None)
X_train_21 = prepr_21.fit_transform(X_train)
X_val_21 = prepr_21.transform(X_val)
X_test_21 = prepr_21.transform(test_texts)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_21 = svd.fit_transform(X_train_21)
X_test_21 = svd.transform(X_test_21)
X_val_21 = svd.transform(X_val_21)

prepr_22 = Prepr(p_type='lemmatization', vectorizer='frequency', filter_pos=None)
X_train_22 = prepr_22.fit_transform(X_train)
X_val_22 = prepr_22.transform(X_val)
X_test_22 = prepr_22.transform(test_texts)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_22 = svd.fit_transform(X_train_22)
X_test_22 = svd.transform(X_test_22)
X_val_22 = svd.transform(X_val_22)

prepr_31 = Prepr(p_type='lemmatization', vectorizer='tf-idf', filter_pos=['N', 'J'])
X_train_31 = prepr_31.fit_transform(X_train)
X_val_31 = prepr_31.transform(X_val)
X_test_31 = prepr_31.transform(test_texts)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_31 = svd.fit_transform(X_train_31)
X_test_31 = svd.transform(X_test_31)
X_val_31 = svd.transform(X_val_31)

prepr_32 = Prepr(p_type='lemmatization', vectorizer='binary', filter_pos=['N', 'J'])
X_train_32 = prepr_32.fit_transform(X_train)
X_val_32 = prepr_32.transform(X_val)
X_test_32 = prepr_32.transform(test_texts)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_32 = svd.fit_transform(X_train_32)
X_test_32 = svd.transform(X_test_32)
X_val_32 = svd.transform(X_val_32)

prepr_33 = Prepr(p_type='lemmatization', vectorizer='frequency', filter_pos=['N', 'J'])
X_train_33 = prepr_33.fit_transform(X_train)
X_val_33 = prepr_33.transform(X_val)
X_test_33 = prepr_33.transform(test_texts)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_33 = svd.fit_transform(X_train_33)
X_test_33 = svd.transform(X_test_33)
X_val_33 = svd.transform(X_val_33)
def model_train(X_train, y_train, X_val, y_val):
    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    results = {}

    dt = DecisionTreeClassifier(
        class_weight='balanced',
        max_depth=20,
        min_samples_leaf=20,
        min_samples_split=100,
        random_state=42
    )
    dt.fit(X_train_scaled, y_train)
    y_pred_dt = dt.predict(X_val_scaled)
    results['DecisionTree'] = {
        'macro': f1_score(y_val, y_pred_dt, average='macro'),
        'micro': f1_score(y_val, y_pred_dt, average='micro'),
        'weighted': f1_score(y_val, y_pred_dt, average='weighted')
    }

    rf = RandomForestClassifier(
        class_weight='balanced',
        max_depth=None,
        min_samples_leaf=10,
        min_samples_split=20,
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_scaled, y_train)
    y_pred_rf = rf.predict(X_val_scaled)
    results['RandomForest'] = {
        'macro': f1_score(y_val, y_pred_rf, average='macro'),
        'micro': f1_score(y_val, y_pred_rf, average='micro'),
        'weighted': f1_score(y_val, y_pred_rf, average='weighted')
    }

    gb = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        random_state=42,
        eval_metric='mlogloss'
    )
    gb.fit(X_train_scaled, y_train)
    y_pred_gb = gb.predict(X_val_scaled)
    results['XGBoost'] = {
        'macro': f1_score(y_val, y_pred_gb, average='macro'),
        'micro': f1_score(y_val, y_pred_gb, average='micro'),
        'weighted': f1_score(y_val, y_pred_gb, average='weighted')
    }

    base_estimator = LogisticRegression(
    max_iter=100,
    class_weight='balanced',
    random_state=42,
    C=1.0
    )
    ab = AdaBoostClassifier(
        estimator=base_estimator,
        n_estimators=100,
        learning_rate=1.0,
        random_state=42
    )
    ab.fit(X_train_scaled, y_train)
    y_pred_ab = ab.predict(X_val_scaled)
    results['AdaBoost'] = {
        'macro': f1_score(y_val, y_pred_ab, average='macro'),
        'micro': f1_score(y_val, y_pred_ab, average='micro'),
        'weighted': f1_score(y_val, y_pred_ab, average='weighted')
    }

    return results

method_descriptions = {
    '11': 'Лемматизация + TF-IDF',
    '12': 'Стемминг + TF-IDF',
    '21': 'Лемматизация + Binary',
    '22': 'Лемматизация + Frequency',
    '31': 'Лемматизация + сущ./прил. + TF-IDF',
    '32': 'Лемматизация + сущ./прил. + Binary',
    '33': 'Лемматизация + сущ./прил. + Frequency'
}

data_dict = {
    '11': (X_train_11, X_val_11, X_test_11),
    '12': (X_train_12, X_val_12, X_test_12),
    '21': (X_train_21, X_val_21, X_test_21),
    '22': (X_train_22, X_val_22, X_test_22),
    '31': (X_train_31, X_val_31, X_test_31),
    '32': (X_train_32, X_val_32, X_test_32),
    '33': (X_train_33, X_val_33, X_test_33)
}

all_results = []

for method_name, (X_train_data, X_val_data, X_test_data) in data_dict.items():
    print(f"\n{method_name}: {method_descriptions[method_name]}")

    results = model_train(X_train_data, y_train, X_val_data, y_val)

    for model_name, metrics in results.items():
        all_results.append({
            'Method': method_name,
            'Description': method_descriptions[method_name],
            'Model': model_name,
            'F1_macro': metrics['macro'],
            'F1_micro': metrics['micro'],
            'F1_weighted': metrics['weighted']
        })
        print(f"  {model_name}: macro={metrics['macro']:.4f}, micro={metrics['micro']:.4f}")

df_results = pd.DataFrame(all_results)

pivot_micro = df_results.pivot_table(
    values='F1_micro',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

pivot_macro = df_results.pivot_table(
    values='F1_macro',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

pivot_weighted = df_results.pivot_table(
    values='F1_weighted',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

print("\nF1 MICRO:")
print(pivot_micro)

print("\nF1 MACRO:")
print(pivot_macro)

print("\nF1 WEIGHTED:")
print(pivot_weighted)

best_micro = df_results.loc[df_results.groupby('Model')['F1_micro'].idxmax()]

for _, row in best_micro.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  Метод: {row['Description']}")
    print(f"  F1_micro: {row['F1_micro']:.4f}")
    print(f"  F1_macro: {row['F1_macro']:.4f}")

for model_name in ['DecisionTree', 'RandomForest', 'XGBoost', 'AdaBoost']:
    best_row = df_results[df_results['Model'] == model_name].loc[
        df_results[df_results['Model'] == model_name]['F1_micro'].idxmax()
    ]
    best_method = best_row['Method']
    best_desc = best_row['Description']
    print(f"\n{model_name} (лучший: {best_desc})")
    X_train_full = data_dict[best_method][0]
    X_test_data = data_dict[best_method][2]

    from scipy.sparse import vstack
    X_train_combined = vstack([X_train_full, data_dict[best_method][1]])
    y_train_combined = np.concatenate([y_train, y_val])

    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train_combined)
    X_test_scaled = scaler.transform(X_test_data)

    if model_name == 'DecisionTree':
        model = DecisionTreeClassifier(
            class_weight='balanced',
            max_depth=20,
            min_samples_leaf=20,
            min_samples_split=100,
            random_state=42
        )
    elif model_name == 'RandomForest':
        model = RandomForestClassifier(
            class_weight='balanced',
            max_depth=None,
            min_samples_leaf=10,
            min_samples_split=20,
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    elif model_name == 'XGBoost':
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0,
            reg_alpha=0,
            reg_lambda=1,
            random_state=42,
            eval_metric='mlogloss'
        )
    else:
        base_estimator = DecisionTreeClassifier(max_depth=1, class_weight='balanced')
        model = AdaBoostClassifier(
            estimator=base_estimator,
            n_estimators=100,
            learning_rate=1.0,
            random_state=42
        )

    model.fit(X_train_scaled, y_train_combined)
    y_pred_test = model.predict(X_test_scaled)

    test_micro = f1_score(test_labels, y_pred_test, average='micro')
    test_macro = f1_score(test_labels, y_pred_test, average='macro')

    print(f"  Test F1_micro: {test_micro:.4f}")
    print(f"  Test F1_macro: {test_macro:.4f}")


11: Лемматизация + TF-IDF
  DecisionTree: macro=0.2845, micro=0.3025
  RandomForest: macro=0.4729, micro=0.5328
  XGBoost: macro=0.4343, micro=0.5459
  AdaBoost: macro=0.3698, micro=0.4062

12: Стемминг + TF-IDF
  DecisionTree: macro=0.2759, micro=0.2969
  RandomForest: macro=0.4560, micro=0.5119
  XGBoost: macro=0.4323, micro=0.5434
  AdaBoost: macro=0.4004, micro=0.4412

21: Лемматизация + Binary
  DecisionTree: macro=0.1827, micro=0.1916
  RandomForest: macro=0.2969, micro=0.3887
  XGBoost: macro=0.2505, micro=0.4128
  AdaBoost: macro=0.2454, micro=0.3416

22: Лемматизация + Frequency
  DecisionTree: macro=0.1702, micro=0.1800
  RandomForest: macro=0.2966, micro=0.3903
  XGBoost: macro=0.2601, micro=0.4222
  AdaBoost: macro=0.2413, micro=0.2731

31: Лемматизация + сущ./прил. + TF-IDF
  DecisionTree: macro=0.3064, micro=0.3178
  RandomForest: macro=0.4985, micro=0.5609
  XGBoost: macro=0.5013, micro=0.6016
  AdaBoost: macro=0.3770, micro=0.4194

32: Лемматизация + сущ./прил. + Binar

KeyboardInterrupt: 

Задания 6, 7 и 8:
🔶 1. Сейчас у вас в работе 4 классификатора (см
 задание 5). По каждому смотрим какие параметры есть и эксперементируем с их настройками.
Заодно разбираемся, что такое настройка гиперпараметров и как их настраивать автоматически:
см.далее.

🔶2. Для этих классификатров, кроме настроек гиперпараметров, эксперементируем с представлением текста:
2.1 - пробуем использовать Латентно-семантический анализ (LSA)  (вспоминаем первые лекции). Эксперементируем с разным количеством латентных тем - это количество задает длину вектора представляющего документ после LSA. На вход LSA подаете матрицу векторов tf-idf. Смотрим, получается ли улучшить качество классификации в сравнении с использованием для представления текстов векторов tf-idf по всем словам словаря коллекции текстов. Какие бонусы потенциально есть у LSA?

2.2 смотрим, можно ли улучшить качество классификации, если в текстах оставлять толтко сущ. и прил. или другие части речи. Можно попробовать обьединить с LSA.

Для обоих датасетов.

В целом это небольшое соревнование☀️ первые 3 лучшие результата отметим и попробуем повторить. В конце занятия, присылаеие информацию какие лучшие значения оценки качества кластеризации удалось получить. Кроме этого можно продолдить экспепементировать дома и обновить оценку.
Для оценки используем micro-averave F1-score

3 модельки посмотрю более вероятные RandomForest, XGBoost, AdaBoost с Лемматизация + Frequency (счет)

In [30]:
dataset = load_dataset('emotion')
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])
validation_df = pd.DataFrame(dataset['validation'])
X_train = list(df_train['text']) + list(validation_df['text'])
y_train = list(df_train['label']) + list(validation_df['label'])
X_test = list(df_test['text'])
y_test = list(df_test['label'])
prepr = Prepr(p_type='lemmatization',vectorizer='frequency')
X_train = prepr.fit_transform(X_train)
X_test = prepr.transform(X_test)

In [31]:
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [32]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [79]:
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_leaf=2, min_samples_split=5,
                       n_estimators=500, n_jobs=-1, random_state=42)

In [71]:
0.8345

0.8345

In [80]:
y_pred = rf.predict(X_test)
f1_score(y_test, y_pred, average='micro')

0.7635

0.888

In [36]:
gb = xgb.XGBClassifier(
    n_estimators=303,
    max_depth=7,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    eval_metric='mlogloss',
    early_stopping_rounds=20
    )

In [37]:
gb.fit(X_train,y_train,eval_set=[(X_test,y_test)])

[0]	validation_0-mlogloss:1.48340
[1]	validation_0-mlogloss:1.42978
[2]	validation_0-mlogloss:1.38517
[3]	validation_0-mlogloss:1.35113
[4]	validation_0-mlogloss:1.32051
[5]	validation_0-mlogloss:1.29078
[6]	validation_0-mlogloss:1.26366
[7]	validation_0-mlogloss:1.24087
[8]	validation_0-mlogloss:1.21941
[9]	validation_0-mlogloss:1.19983
[10]	validation_0-mlogloss:1.18009
[11]	validation_0-mlogloss:1.16061
[12]	validation_0-mlogloss:1.14252
[13]	validation_0-mlogloss:1.12512
[14]	validation_0-mlogloss:1.10924
[15]	validation_0-mlogloss:1.09415
[16]	validation_0-mlogloss:1.08090
[17]	validation_0-mlogloss:1.06743
[18]	validation_0-mlogloss:1.05512
[19]	validation_0-mlogloss:1.04313
[20]	validation_0-mlogloss:1.02875
[21]	validation_0-mlogloss:1.01593
[22]	validation_0-mlogloss:1.00619
[23]	validation_0-mlogloss:0.99550
[24]	validation_0-mlogloss:0.98702
[25]	validation_0-mlogloss:0.97673
[26]	validation_0-mlogloss:0.96594
[27]	validation_0-mlogloss:0.95514
[28]	validation_0-mlogloss:0.9

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=303, n_jobs=None,
              num_parallel_tree=None, ...)

In [38]:
y_pred = gb.predict(X_test)
f1_score(y_test, y_pred, average='micro')

0.888

In [57]:

base_lr = LogisticRegression(
    max_iter=100,
    class_weight='balanced',
    random_state=42,
    C=1.0
)

ab = AdaBoostClassifier(
    estimator=base_lr,
    n_estimators=50,
    learning_rate=0.5,
    random_state=42
)


In [58]:
ab.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


AdaBoostClassifier(algorithm='SAMME',
                   estimator=LogisticRegression(class_weight='balanced',
                                                random_state=42),
                   learning_rate=0.5, random_state=42)

In [59]:
y_pred = ab.predict(X_test)
f1_score(y_pred, y_test, average='micro')

0.778

In [15]:

from sklearn.datasets import fetch_20newsgroups

categories = [
    'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware',
    'comp.graphics',
    'comp.windows.x'
]

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

all_texts = newsgroups.data
all_labels = newsgroups.target
X_train_full, X_test, y_train_full, y_test = train_test_split(
    all_texts, all_labels,
    test_size=0.2,
    random_state=42,
    stratify=all_labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)
prepr_11 = Prepr(p_type='lemmatization', vectorizer='tf-idf', filter_pos=None)
X_train_11 = prepr_11.fit_transform(X_train)
X_val_11 = prepr_11.transform(X_val)
X_test_11 = prepr_11.transform(X_test)
print('1')
svd_11 = TruncatedSVD(n_components=100, random_state=42)
X_train_11 = svd_11.fit_transform(X_train_11)
X_val_11 = svd_11.transform(X_val_11)
X_test_11 = svd_11.transform(X_test_11)

prepr_12 = Prepr(p_type='stemming', vectorizer='tf-idf', filter_pos=None)
X_train_12 = prepr_12.fit_transform(X_train)
X_val_12 = prepr_12.transform(X_val)
X_test_12 = prepr_12.transform(X_test)

svd_12 = TruncatedSVD(n_components=100, random_state=42)
X_train_12 = svd_12.fit_transform(X_train_12)
X_val_12 = svd_12.transform(X_val_12)
X_test_12 = svd_12.transform(X_test_12)
print('2')
prepr_21 = Prepr(p_type='lemmatization', vectorizer='binary', filter_pos=None)
X_train_21 = prepr_21.fit_transform(X_train)
X_val_21 = prepr_21.transform(X_val)
X_test_21 = prepr_21.transform(X_test)

svd_21 = TruncatedSVD(n_components=100, random_state=42)
X_train_21 = svd_21.fit_transform(X_train_21)
X_val_21 = svd_21.transform(X_val_21)
X_test_21 = svd_21.transform(X_test_21)
print('3')
prepr_22 = Prepr(p_type='lemmatization', vectorizer='frequency', filter_pos=None)
X_train_22 = prepr_22.fit_transform(X_train)
X_val_22 = prepr_22.transform(X_val)
X_test_22 = prepr_22.transform(X_test)

svd_22 = TruncatedSVD(n_components=100, random_state=42)
X_train_22 = svd_22.fit_transform(X_train_22)
X_val_22 = svd_22.transform(X_val_22)
X_test_22 = svd_22.transform(X_test_22)
print('4')
prepr_31 = Prepr(p_type='lemmatization', vectorizer='tf-idf', filter_pos=['N', 'J'])
X_train_31 = prepr_31.fit_transform(X_train)
X_val_31 = prepr_31.transform(X_val)
X_test_31 = prepr_31.transform(X_test)

svd_31 = TruncatedSVD(n_components=100, random_state=42)
X_train_31 = svd_31.fit_transform(X_train_31)
X_val_31 = svd_31.transform(X_val_31)
X_test_31 = svd_31.transform(X_test_31)
print('5')
prepr_32 = Prepr(p_type='lemmatization', vectorizer='binary', filter_pos=['N', 'J'])
X_train_32 = prepr_32.fit_transform(X_train)
X_val_32 = prepr_32.transform(X_val)
X_test_32 = prepr_32.transform(X_test)
print('6')
svd_32 = TruncatedSVD(n_components=100, random_state=42)
X_train_32 = svd_32.fit_transform(X_train_32)
X_val_32 = svd_32.transform(X_val_32)
X_test_32 = svd_32.transform(X_test_32)

prepr_33 = Prepr(p_type='lemmatization', vectorizer='frequency', filter_pos=['N', 'J'])
X_train_33 = prepr_33.fit_transform(X_train)
X_val_33 = prepr_33.transform(X_val)
X_test_33 = prepr_33.transform(X_test)
print('7')
svd_33 = TruncatedSVD(n_components=100, random_state=42)
X_train_33 = svd_33.fit_transform(X_train_33)
X_val_33 = svd_33.transform(X_val_33)
X_test_33 = svd_33.transform(X_test_33)

In [16]:
def model_train(X_train, y_train, X_val, y_val):
    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    results = {}
    dt = DecisionTreeClassifier(
        class_weight='balanced',
        max_depth=20,
        min_samples_leaf=20,
        min_samples_split=100,
        random_state=42
    )
    dt.fit(X_train_scaled, y_train)
    y_pred_dt = dt.predict(X_val_scaled)
    results['DecisionTree'] = {
        'macro': f1_score(y_val, y_pred_dt, average='macro'),
        'micro': f1_score(y_val, y_pred_dt, average='micro'),
        'weighted': f1_score(y_val, y_pred_dt, average='weighted')
    }
    rf = RandomForestClassifier(
        class_weight='balanced',
        max_depth=None,
        min_samples_leaf=10,
        min_samples_split=20,
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_scaled, y_train)
    y_pred_rf = rf.predict(X_val_scaled)
    results['RandomForest'] = {
        'macro': f1_score(y_val, y_pred_rf, average='macro'),
        'micro': f1_score(y_val, y_pred_rf, average='micro'),
        'weighted': f1_score(y_val, y_pred_rf, average='weighted')
    }
    gb = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        random_state=42,
        eval_metric='mlogloss'
    )
    gb.fit(X_train_scaled, y_train)
    y_pred_gb = gb.predict(X_val_scaled)
    results['XGBoost'] = {
        'macro': f1_score(y_val, y_pred_gb, average='macro'),
        'micro': f1_score(y_val, y_pred_gb, average='micro'),
        'weighted': f1_score(y_val, y_pred_gb, average='weighted')
    }
    base_estimator = LogisticRegression(
          max_iter=100,
          class_weight='balanced',
          random_state=42,
          C=1.0
        )
    ab = AdaBoostClassifier(
            estimator=base_estimator,
            n_estimators=100,
            learning_rate=1.0,
            random_state=42
        )
    ab.fit(X_train_scaled, y_train)
    y_pred_ab = ab.predict(X_val_scaled)
    results['AdaBoost'] = {
            'macro': f1_score(y_val, y_pred_ab, average='macro'),
            'micro': f1_score(y_val, y_pred_ab, average='micro'),
            'weighted': f1_score(y_val, y_pred_ab, average='weighted')
        }
    return results
method_descriptions = {
    '11': 'Лемматизация + TF-IDF',
    '12': 'Стемминг + TF-IDF',
    '21': 'Лемматизация + Binary',
    '22': 'Лемматизация + Frequency',
    '31': 'Лемматизация + сущ./прил. + TF-IDF',
    '32': 'Лемматизация + сущ./прил. + Binary',
    '33': 'Лемматизация + сущ./прил. + Frequency'
}

data_dict = {
    '11': (X_train_11, X_val_11, X_test_11),
    '12': (X_train_12, X_val_12, X_test_12),
    '21': (X_train_21, X_val_21, X_test_21),
    '22': (X_train_22, X_val_22, X_test_22),
    '31': (X_train_31, X_val_31, X_test_31),
    '32': (X_train_32, X_val_32, X_test_32),
    '33': (X_train_33, X_val_33, X_test_33)
}

all_results = []

for method_name, (X_train_data, X_val_data, X_test_data) in data_dict.items():
    print(f"\n{method_name}: {method_descriptions[method_name]}")

    results = model_train(X_train_data, y_train, X_val_data, y_val)

    for model_name, metrics in results.items():
        all_results.append({
            'Method': method_name,
            'Description': method_descriptions[method_name],
            'Model': model_name,
            'F1_macro': metrics['macro'],
            'F1_micro': metrics['micro'],
            'F1_weighted': metrics['weighted']
        })
        print(f"  {model_name}: macro={metrics['macro']:.4f}, micro={metrics['micro']:.4f}")
df_results = pd.DataFrame(all_results)

pivot_micro = df_results.pivot_table(
    values='F1_micro',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

pivot_macro = df_results.pivot_table(
    values='F1_macro',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

pivot_weighted = df_results.pivot_table(
    values='F1_weighted',
    index='Description',
    columns='Model',
    aggfunc='first'
).round(4)

print("\nF1 MICRO:")
print(pivot_micro)

print("\nF1 MACRO:")
print(pivot_macro)

print("\nF1 WEIGHTED:")
print(pivot_weighted)
best_micro = df_results.loc[df_results.groupby('Model')['F1_micro'].idxmax()]

for _, row in best_micro.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  Метод: {row['Description']}")
    print(f"  F1_micro: {row['F1_micro']:.4f}")
    print(f"  F1_macro: {row['F1_macro']:.4f}")

from scipy.sparse import vstack

for model_name in ['DecisionTree', 'RandomForest', 'XGBoost', 'AdaBoost']:
    if model_name == 'AdaBoost':
        ada_scores = df_results[df_results['Model'] == 'AdaBoost']['F1_micro']
        if ada_scores.max() == 0:
            print(f"\n{model_name} пропущен (не обучился)")
            continue

    best_row = df_results[df_results['Model'] == model_name].loc[
        df_results[df_results['Model'] == model_name]['F1_micro'].idxmax()
    ]
    best_method = best_row['Method']
    best_desc = best_row['Description']

    print(f"\n{model_name} (лучший: {best_desc})")
    X_train_full = data_dict[best_method][0]
    X_test_data = data_dict[best_method][2]
    X_train_combined = vstack([X_train_full, data_dict[best_method][1]])
    y_train_combined = np.concatenate([y_train, y_val])
    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train_combined)
    X_test_scaled = scaler.transform(X_test_data)
    if model_name == 'DecisionTree':
        model = DecisionTreeClassifier(
            class_weight='balanced',
            max_depth=20,
            min_samples_leaf=20,
            min_samples_split=100,
            random_state=42
        )
    elif model_name == 'RandomForest':
        model = RandomForestClassifier(
            class_weight='balanced',
            max_depth=None,
            min_samples_leaf=10,
            min_samples_split=20,
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    elif model_name == 'XGBoost':
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0,
            reg_alpha=0,
            reg_lambda=1,
            random_state=42,
            eval_metric='mlogloss'
        )
    else:
        base_estimator = LogisticRegression(
            max_iter=100,
            class_weight='balanced',
            random_state=42
        )
        model = AdaBoostClassifier(
            estimator=base_estimator,
            n_estimators=100,
            learning_rate=1.0,
            random_state=42
        )
    model.fit(X_train_scaled, y_train_combined)
    y_pred_test = model.predict(X_test_scaled)

    test_micro = f1_score(y_test, y_pred_test, average='micro')
    test_macro = f1_score(y_test, y_pred_test, average='macro')
    test_weighted = f1_score(y_test, y_pred_test, average='weighted')

    print(f"  Test F1_micro: {test_micro:.4f}")
    print(f"  Test F1_macro: {test_macro:.4f}")
    print(f"  Test F1_weighted: {test_weighted:.4f}")


11: Лемматизация + TF-IDF
  DecisionTree: macro=0.5989, micro=0.6032
  RandomForest: macro=0.7071, micro=0.7072
  XGBoost: macro=0.7310, micro=0.7312
  AdaBoost: macro=0.6728, micro=0.6752

12: Стемминг + TF-IDF
  DecisionTree: macro=0.6087, micro=0.6096
  RandomForest: macro=0.7004, micro=0.7024
  XGBoost: macro=0.7194, micro=0.7216
  AdaBoost: macro=0.6912, micro=0.6912

21: Лемматизация + Binary
  DecisionTree: macro=0.5394, micro=0.5424
  RandomForest: macro=0.6525, micro=0.6544
  XGBoost: macro=0.7051, micro=0.7072
  AdaBoost: macro=0.6770, micro=0.6768

22: Лемматизация + Frequency
  DecisionTree: macro=0.5409, micro=0.5408
  RandomForest: macro=0.6774, micro=0.6800
  XGBoost: macro=0.7144, micro=0.7168
  AdaBoost: macro=0.5999, micro=0.5984

31: Лемматизация + сущ./прил. + TF-IDF
  DecisionTree: macro=0.6323, micro=0.6336
  RandomForest: macro=0.7195, micro=0.7200
  XGBoost: macro=0.7118, micro=0.7120
  AdaBoost: macro=0.6863, micro=0.6864

32: Лемматизация + сущ./прил. + Binar

In [18]:
newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

all_texts = newsgroups.data
all_labels = newsgroups.target
X_train_full, X_test, y_train_full, y_test = train_test_split(
    all_texts, all_labels,
    test_size=0.2,
    random_state=42,
    stratify=all_labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

In [19]:
prepr_11 = Prepr(p_type='lemmatization', vectorizer='tf-idf', filter_pos=None)
X_train_11 = prepr_11.fit_transform(X_train)
X_test_11 = prepr_11.transform(X_test)


In [21]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_11)
X_test = scaler.transform(X_test_11)

In [28]:
gb = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        random_state=42,
        eval_metric='mlogloss'
    )
gb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [29]:
y_pred = gb.predict(X_test)
f1_score(y_pred, y_test, average='micro')

0.7595907928388747